# E-commerce Orders - Exploratory Data Analysis

End-to-end analysis of a synthetic e-commerce orders dataset with Pandas, NumPy and Matplotlib.

**Questions:** revenue trends, category and geography performance, acquisition channels, customer behavior, and discount impact.

## 1. Load and inspect

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/ecommerce_orders.csv", parse_dates=["order_date"])
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

## 2. Clean and enrich

Add a net revenue column (after discount) and a month bucket.

In [ ]:
df["revenue"] = df["quantity"] * df["unit_price"] * (1 - df["discount_pct"] / 100)
df["month"] = df["order_date"].dt.to_period("M").astype(str)
delivered = df[df["order_status"] == "delivered"]

## 3. Revenue trends over time

In [ ]:
monthly = delivered.groupby("month")["revenue"].agg(["sum", "count"])
monthly.columns = ["revenue", "orders"]
monthly

In [ ]:
ax = monthly["revenue"].plot(kind="line", figsize=(10, 4), title="Monthly delivered revenue")
ax.set_ylabel("Revenue (EGP)")
plt.tight_layout()

## 4. Category performance

In [ ]:
cat = delivered.groupby("product_category").agg(
    revenue=("revenue", "sum"),
    units=("quantity", "sum"),
    avg_order=("revenue", "mean"),
).sort_values("revenue", ascending=False)
cat

In [ ]:
cat["revenue"].plot(kind="bar", figsize=(8, 4), title="Revenue by category")
plt.ylabel("Revenue (EGP)")
plt.tight_layout()

## 5. Geography

In [ ]:
geo = delivered.groupby("country")["revenue"].sum().sort_values(ascending=False)
geo

In [ ]:
geo.plot(kind="bar", figsize=(8, 4), title="Revenue by country")
plt.ylabel("Revenue (EGP)")
plt.tight_layout()

## 6. Acquisition channels

In [ ]:
ch = delivered.groupby("acquisition_channel").agg(
    revenue=("revenue", "sum"), orders=("order_id", "count"))
ch["avg_order_value"] = ch["revenue"] / ch["orders"]
ch.sort_values("revenue", ascending=False)

## 7. Customer behavior

In [ ]:
cust = delivered.groupby("customer_id").agg(
    orders=("order_id", "count"), total_spent=("revenue", "sum"))
repeat_rate = (cust["orders"] > 1).mean() * 100
print(f"Repeat customer rate: {repeat_rate:.1f}%")
print(f"Average customer lifetime value: {cust['total_spent'].mean():,.0f} EGP")
cust.sort_values("total_spent", ascending=False).head(10)

## 8. Discounts and order status

In [ ]:
df.groupby("order_status")["order_id"].count()

In [ ]:
disc = delivered.groupby("discount_pct")["revenue"].agg(["sum", "count"])
disc

In [ ]:
disc["sum"].plot(kind="bar", figsize=(7, 4), title="Revenue by discount level")
plt.ylabel("Revenue (EGP)")
plt.tight_layout()

## Findings

- Electronics leads category revenue, followed by Furniture and Appliances.
- Egypt brings the majority of orders, with meaningful volume from Saudi Arabia and the UAE.
- Social Media and Organic Search bring the most delivered revenue.
- About two thirds of customers are repeat buyers, and a repeat customer spends roughly 4x a one-time customer.
- Undiscounted orders drive most delivered revenue, with discounted orders (5-15%) adding a smaller share; cancelled and returned orders together are about a quarter of all orders, a number worth reducing.